In [1]:
import os
import sys
sys.path.append("kaggle/input/polymer_pipeline")

In [2]:
from data_preparation import get_data_paths, load_and_split_data
import model

In [3]:
os.environ['NEURIPS_DATA_PATH']     = 'kaggle/input/neurips-open-polymer-prediction-2025'
os.environ['EXTRA_DATA_BASE']       = 'kaggle/input/smiles-extra-data'
os.environ['TC_DATA_BASE']          = 'kaggle/input/tc-smiles'

In [4]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [5]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

原始训练: 7973 条
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737 | 填充: 0
新增样本: 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526 | 填充: 15
新增样本: 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0 | 填充: 0
新增样本: 499 条
  → 正在增强 Density 数据，共 787 条


[02:49:11] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[02:49:11] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[02:49:11] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[02:49:11] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[02:49:11] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[02:49:11] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[02:49:11] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[02:49:11] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[02:49:11] SMILES Parse 

cross_smiles: 254 | 填充: 110
新增样本: 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43 | 填充: 43
新增样本: 819 条
Loaded: 8064 1008 1009


In [6]:
from train_stage1 import (setup_stage1_data, create_stage1_model, 
                        optimize_stage1, train_final_stage1_model)

/usr/local/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
print(train_df.head())

             id                                             SMILES  Tg  \
0  2.026603e+09    *Oc1ccc(CC2(Cc3ccc(*)cc3)c3ccccc3-c3ccccc32)cc1 NaN   
1  1.189403e+09         *CC(O)COc1ccc(C(C)CC(C)(C)c2ccc(O*)cc2)cc1 NaN   
2  1.686537e+09  *C(=O)c1ccc2c(c1)C(=O)N(c1c(C)cc(C(c3cc(C)c(N4... NaN   
3  2.632934e+08  *Oc1ccc2ccc(Oc3ccc(C(=Nc4ccc(N=C(c5ccccc5)c5cc... NaN   
4  1.280165e+09                    *Nc1ccc(-c2ccc(N*)c(OC)c2)cc1OC NaN   

        FFV  Tc  Density  Rg  
0  0.386736 NaN      NaN NaN  
1  0.354235 NaN      NaN NaN  
2  0.430396 NaN      NaN NaN  
3  0.381740 NaN      NaN NaN  
4  0.334115 NaN      NaN NaN  


In [8]:
# 运行Optuna优化
study = optimize_stage1(
    train_df,
    study_name="new_stage1_graph_ssl",
    n_trials=50,  # 可以根据需要调整
    patience=15
)

# 查看最佳参数
print("Best trial:")
print(" Value: ", study.best_trial.value)
print(" Params: ")
for key, value in study.best_trial.params.items():
    print(f"   {key}: {value}")

📦 构建 PolymerDataset，样本数=8064
   成功转换为图数据: 8064 条


[I 2025-08-02 02:49:27,031] Using an existing study with name 'new_stage1_graph_ssl' instead of creating a new one.


------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------
1       4.0473e+00  inf         ✅ Improved
2       2.4545e+00  1.59e+00    ✅ Improved
3       1.9563e+00  4.98e-01    ✅ Improved
4       1.6039e+00  3.52e-01    ✅ Improved
5       1.3548e+00  2.49e-01    ✅ Improved
6       1.2445e+00  1.10e-01    ✅ Improved
7       1.1779e+00  6.66e-02    ✅ Improved
8       1.1142e+00  6.36e-02    ✅ Improved
9       1.0623e+00  5.20e-02    ✅ Improved
10      1.0127e+00  4.95e-02    ✅ Improved
11      9.6294e-01  4.98e-02    ✅ Improved
12      9.1673e-01  4.62e-02    ✅ Improved
13      8.7009e-01  4.66e-02    ✅ Improved
14      8.1961e-01  5.05e-02    ✅ Improved
15      7.7386e-01  4.57e-02    ✅ Improved
16      7.2787e-01  4.60e-02    ✅ Improved
17      6.8916e-01  3.87e-02    ✅ Improved
18      6.5332e-01  3.58e-02    ✅ Improved
19      6.2006e-01  3.33e-02    ✅ Improved
20      5.8826e-01  3.18e-02    ✅ Improved
21      5.5

[I 2025-08-02 03:02:47,828] Trial 2 finished with value: 0.12712366541936285 and parameters: {'lr': 4.710355603303735e-05, 'hidden_dim': 64, 'num_edge_layers': 4}. Best is trial 2 with value: 0.12712366541936285.


200     1.2776e-01  -6.36e-04   🚫 No improve (1/15)
------------------------------------------
Best loss: 1.2712e-01 at epoch 199
------------------------------------------
------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------
1       3.6636e+00  inf         ✅ Improved
2       1.9901e+00  1.67e+00    ✅ Improved
3       1.5520e+00  4.38e-01    ✅ Improved
4       1.3788e+00  1.73e-01    ✅ Improved
5       1.2886e+00  9.03e-02    ✅ Improved
6       1.2207e+00  6.78e-02    ✅ Improved
7       1.1648e+00  5.59e-02    ✅ Improved
8       1.1101e+00  5.47e-02    ✅ Improved
9       1.0526e+00  5.76e-02    ✅ Improved
10      9.9998e-01  5.26e-02    ✅ Improved
11      9.4488e-01  5.51e-02    ✅ Improved
12      8.8996e-01  5.49e-02    ✅ Improved
13      8.4129e-01  4.87e-02    ✅ Improved
14      7.9402e-01  4.73e-02    ✅ Improved
15      7.5230e-01  4.17e-02    ✅ Improved
16      7.1254e-01  3.98e-02    ✅ Improved
17      6.

...

Epoch 1: Trial pruned
------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------


[I 2025-08-02 05:23:26,848] Trial 50 pruned. 


Epoch 1: Trial pruned
------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------


[I 2025-08-02 05:23:30,588] Trial 51 pruned. 


Epoch 1: Trial pruned
✅ Saved best artifacts to stage1_artifacts:
   - Parameters: stage1_artifacts/stage1_best_params_trial6.pt
   - Encoder: stage1_artifacts/stage1_encoder_best_trial6.pt
⚠️ Found 16 remaining files in tmp_stage1, cleaning up...
✅ Cleaned up temporary directory: tmp_stage1
Best trial:
 Value:  0.033048626878077074
 Params: 
   lr: 0.0010352506858438
   hidden_dim: 256
   num_edge_layers: 2


In [9]:
param_files = [f for f in os.listdir("stage1_artifacts") if f.startswith("stage1_best_params")]
print(param_files)

['stage1_best_params_trial6.pt']


In [10]:
import torch
latest_file = sorted(param_files)[-1]
params = torch.load(os.path.join("stage1_artifacts", latest_file))

In [12]:
model = train_final_stage1_model(
    train_df,
    params=params,  # 可自动从文件加载
    output_path="stage1_final_model.pth",
    n_epochs=100,
    patience=15
)

📦 构建 PolymerDataset，样本数=8064
   成功转换为图数据: 8064 条
------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------
1       9.4513e-01  inf         ✅ Improved
2       3.5900e-01  5.86e-01    ✅ Improved
3       2.9029e-01  6.87e-02    ✅ Improved
4       2.5128e-01  3.90e-02    ✅ Improved
5       2.0397e-01  4.73e-02    ✅ Improved
6       1.8140e-01  2.26e-02    ✅ Improved
7       1.6505e-01  1.63e-02    ✅ Improved
8       1.5034e-01  1.47e-02    ✅ Improved
9       1.4633e-01  4.01e-03    ✅ Improved
10      1.4194e-01  4.38e-03    ✅ Improved
11      1.2971e-01  1.22e-02    ✅ Improved
12      1.1930e-01  1.04e-02    ✅ Improved
13      1.2099e-01  -1.69e-03   🚫 No improve (1/15)
14      1.1430e-01  5.00e-03    ✅ Improved
15      1.0984e-01  4.45e-03    ✅ Improved
16      1.0379e-01  6.06e-03    ✅ Improved
17      1.0388e-01  -9.75e-05   🚫 No improve (1/15)
18      9.4735e-02  9.05e-03    ✅ Improved
19      9.4300e-02  4.35e-04  